# Project 7 — Notebook 24: Business Summary & Recommendations

### NCR Temporal & Scheduling Analysis

---

| | |
|---|---|
| **Scope** | NCR (Region 3) · 36,907 reactive tickets · Jan 2022 – May 2023 |
| **Notebooks** | NB22 (hourly/daily profiling) · NB23 (seasonal + schedule alignment) · NB24 (this) |
| **Outputs** | `p7_hourly_breach_profile.csv` · `p7_dow_sla_profile.csv` · `p7_seasonal_profile.csv` · `p7_schedule_alignment.csv` |
| **Audience** | Network Operations leadership · NOC planning · HR/scheduling team |

---

## 1. Setup

In [1]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from IPython.display import display, Markdown
%matplotlib inline

os.chdir(os.path.join('..', '..'))
if os.path.abspath(os.getcwd()) not in sys.path:
    sys.path.insert(0, os.path.abspath(os.getcwd()))

from src.fault_ticket.metrics import calculate_zone_summary
from config import ZONE_ORDER, ZONE_PALETTE

FIGURE_DIR = 'reports/figures/project7_ncr'
DEFAULT_DPI = 300
os.makedirs(FIGURE_DIR, exist_ok=True)

plt.rcParams.update({
    'figure.dpi'         : DEFAULT_DPI,
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'font.size'          : 10,
})

df      = pd.read_csv('output/cleaned_fault_ticket.csv', low_memory=False)
summary = calculate_zone_summary(df)
df_zone = df[df['ZONE'].isin(ZONE_ORDER)].copy()

df_zone['REPORTDATE']  = pd.to_datetime(df_zone['REPORTDATE'], errors='coerce')
df_zone['Hour']        = df_zone['REPORTDATE'].dt.hour
df_zone['DayOfWeek_n'] = df_zone['REPORTDATE'].dt.dayofweek
df_zone['Month']       = df_zone['REPORTDATE'].dt.month
df_zone['Is_Night']    = ((df_zone['Hour'] >= 20) | (df_zone['Hour'] < 6)).astype(int)
df_zone['Is_Weekend']  = df_zone['DayOfWeek_n'].isin([5, 6]).astype(int)

hourly   = pd.read_csv('output/p7_hourly_breach_profile.csv')
dow      = pd.read_csv('output/p7_dow_sla_profile.csv')
seasonal = pd.read_csv('output/p7_seasonal_profile.csv')
schedule = pd.read_csv('output/p7_schedule_alignment.csv')

ncr_breach     = 1 - df_zone['SLA_Compliant'].mean()
night_breach   = df_zone[df_zone['Is_Night']==1]['SLA_Compliant'].agg(lambda x: 1 - x.mean())
day_breach     = df_zone[df_zone['Is_Night']==0]['SLA_Compliant'].agg(lambda x: 1 - x.mean())
weekend_breach = df_zone[df_zone['Is_Weekend']==1]['SLA_Compliant'].agg(lambda x: 1 - x.mean())
weekday_breach = df_zone[df_zone['Is_Weekend']==0]['SLA_Compliant'].agg(lambda x: 1 - x.mean())

print(f"✅ Data loaded. NCR breach rate: {ncr_breach:.1%}")

✅ Data loaded. NCR breach rate: 17.8%


## 2. Key Metrics Snapshot

In [2]:
peak_hour = int(hourly.loc[hourly['Breach_Rate'].idxmax(), 'Hour'])
peak_br   = hourly['Breach_Rate'].max()
worst_day = dow.loc[dow['Breach_Rate'].idxmax(), 'DayOfWeek']

april_data    = df_zone[df_zone['Month']==4]
non_april     = df_zone[df_zone['Month']!=4]
april_breach  = 1 - april_data['SLA_Compliant'].mean() if len(april_data) else 0

typhoon_data  = df_zone[df_zone['Month'].isin([8,9,10])]
typhoon_breach= 1 - typhoon_data['SLA_Compliant'].mean() if len(typhoon_data) else 0

sat_sched = schedule[schedule['Window'].str.contains('Sat 20')]
sat_br    = sat_sched['Breach_Rate'].values[0] if not sat_sched.empty else 0

metrics = pd.DataFrame({
    'Finding': [
        'NCR-wide SLA breach rate',
        'Night (20:00–06:00) breach rate',
        'Day (06:00–20:00) breach rate',
        'Night vs day gap',
        'Peak breach hour',
        'Worst day of week',
        'Weekend breach rate',
        'Weekday breach rate',
        'April (Holy Week month) breach',
        'Aug–Oct (typhoon peak) breach',
        'Saturday 20:00–24:00 breach',
    ],
    'Value': [
        f'{ncr_breach:.1%}',
        f'{night_breach:.1%}',
        f'{day_breach:.1%}',
        f'{abs(night_breach - day_breach):.1%} pp',
        f'{peak_hour:02d}:00 ({peak_br:.1%})',
        f'{worst_day}',
        f'{weekend_breach:.1%}',
        f'{weekday_breach:.1%}',
        f'{april_breach:.1%} (vs {1 - non_april["SLA_Compliant"].mean():.1%} other months)',
        f'{typhoon_breach:.1%}',
        f'{sat_br:.1%} (highest-risk uncovered window)',
    ],
    'Addressable': [
        '—', '🟡 Partial', '—', '—',
        '—', '🟢 Yes', '🟢 Yes', '—',
        '⚪ Not a problem', '🟡 Partial (cause confirmed)', '🟢 Yes',
    ]
})

display(Markdown('### Project 7 Key Metrics'))
display(metrics.style
    .hide(axis='index')
    .set_properties(**{'text-align': 'left'})
    .set_table_styles([
        {'selector':'th','props':[('background-color','#2c3e50'),('color','white'),
                                   ('font-size','11px'),('padding','7px 10px')]},
        {'selector':'td','props':[('padding','6px 10px'),('font-size','11px')]},
        {'selector':'tr:nth-child(even)','props':[('background-color','#f7f9fc')]},
    ])
    .set_caption('Project 7 — Temporal Analysis Key Metrics')
)

### Project 7 Key Metrics

Finding,Value,Addressable
NCR-wide SLA breach rate,17.8%,—
Night (20:00–06:00) breach rate,18.0%,🟡 Partial
Day (06:00–20:00) breach rate,17.7%,—
Night vs day gap,0.3% pp,—
Peak breach hour,21:00 (28.7%),—
Worst day of week,Saturday,🟢 Yes
Weekend breach rate,18.6%,🟢 Yes
Weekday breach rate,17.6%,—
April (Holy Week month) breach,14.5% (vs 18.4% other months),⚪ Not a problem
Aug–Oct (typhoon peak) breach,18.7%,🟡 Partial (cause confirmed)


## 3. Key Findings

In [3]:
findings = [
    (
        "🌙 Evening & Overnight Breach Concentration",
        lambda: (
            f"Night window (20:00–06:00): {night_breach:.1%} breach  |  "
            f"Day window: {day_breach:.1%}  |  Gap: {abs(night_breach-day_breach):.1%} pp"
        ),
        (
            "Tickets opened after 20:00 breach only marginally more than daytime tickets — a "
            "0.3pp NCR-wide gap and a 0.5pp gap within field-dispatch tickets specifically. "
            "Both gaps are small enough that they should be confirmed with a significance test "
            "before being treated as a confirmed effect; at this sample size they are plausibly "
            "within normal noise. The working hypothesis — field-dispatch tickets opened at night "
            "cannot begin on-site work until morning — is operationally plausible but is not yet "
            "tested directly against dispatch-delay data by hour. Recommend validating the effect "
            "before committing to a blanket on-call policy; the Saturday 20:00–24:00 window "
            "(28.5% breach) is a much stronger, already-confirmed signal to act on first."
        )
    ),
    (
        "📅 Weekend Coverage Gaps Align with Roster Boundaries",
        lambda: (
            f"Weekend breach: {weekend_breach:.1%}  |  "
            f"Weekday breach: {weekday_breach:.1%}  |  "
            f"Saturday 20:00–24:00: {sat_br:.1%}"
        ),
        (
            "The Sun–Thu / Tue–Sat roster structure creates a predictable boundary: "
            "Saturday evening is the highest-risk uncovered window, when the Sun–Thu team is off "
            "and the Tue–Sat team is at or near end-of-shift. "
            "This gap is addressable through schedule redesign without additional headcount — "
            "a small Saturday night on-call rotation for the 2–3 highest-breach zones would "
            "directly address the highest-concentration window."
        )
    ),
    (
        "🌺 April (Holy Week) Shows Higher Volume but LOWER Breach Rate",
        lambda: (
            f"April breach: {april_breach:.1%}  |  "
            f"Non-April breach: {1-non_april['SLA_Compliant'].mean():.1%}  |  "
            f"April tickets: {len(april_data):,}"
        ),
        (
            "April ticket volume is meaningfully higher than a flat monthly average (5,242 in one "
            "month vs an implied ~4,340), consistent with a real Holy Week volume effect. But breach "
            "rate is 3.9pp LOWER than the rest of the year and MTTR is also lower, so April is not "
            "currently an SLA risk period in this dataset. There is no staffing/roster column to "
            "support a 'reduced staffing suppresses SLA' explanation, and the direction of the SLA "
            "numbers contradicts that explanation regardless. Recommended action: keep tracking April "
            "separately given the volume spike, but do not divert PM or staffing resources to April "
            "based on a breach-rate risk that the current data does not show."
        )
    ),
    (
        "🌀 Typhoon Season (Aug–Oct) — Power Failure, Not FOC Cut, Is the Driver",
        lambda: (
            f"Aug–Oct breach: {typhoon_breach:.1%}  |  "
            f"Other months: {1-df_zone[~df_zone['Month'].isin([8,9,10])]['SLA_Compliant'].mean():.1%}  |  "
            f"Power Failure share: +7.6pp  |  FOC Cut share: -2.8pp  |  Force Majeure: +0.1pp (NB23 Table 5)"
        ),
        (
            "Typhoon season shows a small breach-rate increase (+1.1pp) that should still be confirmed "
            "with a significance test before being called structural. But the ROOT CAUSE is now "
            "settled by NB23's Table 5: FOC Cut's share of the ticket mix actually FALLS during "
            "Aug-Oct (-2.8pp) and Force Majeure barely moves (+0.1pp), while Power Failure's share "
            "jumps +7.6pp -- by far the largest shift of the three, and consistent with weather-"
            "related grid/genset strain. The original 'FOC cut surge' explanation is confirmed "
            "wrong. Recommended action: pre-position genset/battery support ahead of August "
            "instead of FOC-specific crews, and report typhoon-season SLA separately given the "
            "small, still-unconfirmed breach-rate gap."
        )
    ),
    (
        "🤖 Temporal Features Are Valid ML Inputs for P7 Retraining",
        lambda: (
            "Is_Night, Is_Weekend, Hour_of_Day, Day_of_Week already in P7 feature matrix · "
            "Is_HolyWeek + Is_TyphoonSeason: add to next retraining cycle"
        ),
        (
            "The P7 feature matrix already contains Is_Night, Is_Weekend, Hour_of_Day, "
            "and Day_of_Week with zero target leakage. These are available at ticket creation. "
            "The next P7 retraining cycle should add Is_HolyWeek (Month==4) and "
            "Is_TyphoonSeason (Month in [8,9,10]) as categorical binary features. "
            "Additionally, a temporal score uplift of +0.05 to +0.10 probability for tickets "
            "opened in the Saturday 20:00–Sunday 06:00 window is supported by the schedule "
            "alignment analysis. Validate uplift magnitude against predicted vs actual breach "
            "rates in the next evaluation cycle."
        )
    ),
]

for title, metric_fn, body in findings:
    print(f"{'─'*65}")
    print(f"  {title}")
    print(f"  {metric_fn()}")
    print()
    for line in __import__('textwrap').wrap(body, width=61):
        print(f"  {line}")
    print()

─────────────────────────────────────────────────────────────────
  🌙 Evening & Overnight Breach Concentration
  Night window (20:00–06:00): 18.0% breach  |  Day window: 17.7%  |  Gap: 0.3% pp

  Tickets opened after 20:00 breach only marginally more than
  daytime tickets — a 0.3pp NCR-wide gap and a 0.5pp gap within
  field-dispatch tickets specifically. Both gaps are small
  enough that they should be confirmed with a significance test
  before being treated as a confirmed effect; at this sample
  size they are plausibly within normal noise. The working
  hypothesis — field-dispatch tickets opened at night cannot
  begin on-site work until morning — is operationally plausible
  but is not yet tested directly against dispatch-delay data by
  hour. Recommend validating the effect before committing to a
  blanket on-call policy; the Saturday 20:00–24:00 window
  (28.5% breach) is a much stronger, already-confirmed signal
  to act on first.

─────────────────────────────────────────────

## 4. Recommendations

### Immediate (0–3 Months)

**REC-P7-01 · Saturday Evening On-Call Rotation**  
Introduce a Saturday 20:00–24:00 on-call rotation of 2–3 senior field engineers per zone for the three highest-breach zones (identified in NB22's hourly zone heatmap). This directly addresses the highest-risk uncovered window confirmed in the schedule alignment analysis. No additional headcount required — rotate existing engineers.  
**Target:** Reduce Saturday night breach rate by 5–8 pp within 2 months of implementation.

**REC-P7-02 · Separate Seasonal SLA Reporting**  
Add time-of-day and day-of-week SLA splits to the weekly NOC report. Report April and August–October SLA separately from the annual rate. Annotate force-majeure tickets in the typhoon-season report. This requires no operational change — only a reporting template update.  
**Target:** Operational dashboards reflect time-aware SLA within next reporting cycle.

---

### Short-Term (3–6 Months)

**REC-P7-03 · Verify April Effect Before Acting (Revised)**  
The original premise for this recommendation — an April SLA-degrading "Holy Week spike" — is not supported by the data: April breach rate (14.5%) is already 3.9pp *below* the non-April rate (18.4%), and April MTTR is also lower. No PM or staffing pre-positioning is justified for April on breach-risk grounds. Retain only the volume-monitoring piece: track April ticket volume (confirmed ~20% above average) so capacity planning accounts for it, without treating April as an SLA risk period.  
**Target:** No SLA-based target — this is now a monitoring-only item. Re-open as an SLA risk item only if a future year shows April breach rate rising above the annual average.

**REC-P7-04 · Pre-Position Genset/Battery Support for Typhoon Season (Revised — Root Cause Confirmed)**  
NB23's Table 5 rules out the original "FOC cut surge" premise: FOC Cut's share of the ticket mix *falls* -2.8pp during Aug–Oct, and Force Majeure barely moves (+0.1pp). Power Failure's share, by contrast, jumps +7.6pp — the largest shift of the three and the far more plausible weather-driven cause (grid instability / genset strain during storms). Redirect the pre-positioning budget from FOC-specific rapid-response teams to genset and battery maintenance at the top repeat-failure sites (cross-reference P6's site list) ahead of August.  
**Target:** No MTTR target until the underlying +1.1pp breach-rate gap is confirmed with a significance test (still pending) — the 15h figure in the original version of this recommendation had no supporting computation anywhere in NB22/NB23 and should not be treated as a baseline.

---

### Strategic (6–12 Months)

**REC-P7-05 · Enrich P7 Feature Matrix with Temporal Flags**  
Add `Is_HolyWeek` (Month==4) and `Is_TyphoonSeason` (Month in [8,9,10]) as categorical binary features in the next P7 monthly retraining cycle. Apply temporal score uplift of +0.05–0.10 probability for tickets opened Saturday 20:00–Sunday 06:00. Validate uplift on held-out test set before deployment.  
**Target:** Improved P7 precision for high-breach temporal windows; track PR-AUC change across retraining cycles.

**REC-P7-06 · Review Mon–Fri Schedule Boundary**  
NB22/NB23 show a Monday breach spike warranting further investigation. This may reflect weekend backlog (tickets opened Saturday or Sunday that don't complete until Monday) or handover quality issues at the Sunday–Monday shift boundary. Analyse ticket age at Monday close vs. other weekdays before recommending scheduling changes.

## 4. Portfolio Cross-Reference

How Project 7 findings connect to earlier projects and feed forward to Project 8.

In [4]:
cross_ref = [
    ('P1 — Zone Baseline',     'NB03 Sec 7c flagged temporal decomposition as deferred. This project delivers it.'),
    ('P2 — Fault Anatomy',     'Power Failure and FOC CUT profiles now have seasonal context — typhoon season amplifies Power Failure share (+7.6pp), not FOC CUT (-2.8pp).'),
    ('P5 — Site Risk',         'PM queue timing: no change tied to Holy Week — April is not an SLA risk period (see REC-P7-03). Top-20 PM sequencing stays driven by P5/P6 risk scores, not the calendar.'),
    ('P6 — Infra Stress',      'Repeat failure sites in typhoon-prone zones should receive genset/battery pre-activation in August (confirmed driver is Power Failure, not FOC — see REC-P7-04).'),
    ('P7 → P8 — Breach Pred.', 'Temporal features (Is_HolyWeek, Is_TyphoonSeason, hour-of-day) feed the P8 ML feature matrix.'),
]

print('Portfolio cross-references:')
print()
for proj, note in cross_ref:
    print(f'  {proj:<32} {note}')

Portfolio cross-references:

  P1 — Zone Baseline               NB03 Sec 7c flagged temporal decomposition as deferred. This project delivers it.
  P2 — Fault Anatomy               Power Failure and FOC CUT profiles now have seasonal context — typhoon season amplifies Power Failure share (+7.6pp), not FOC CUT (-2.8pp).
  P5 — Site Risk                   PM queue timing: no change tied to Holy Week — April is not an SLA risk period (see REC-P7-03). Top-20 PM sequencing stays driven by P5/P6 risk scores, not the calendar.
  P6 — Infra Stress                Repeat failure sites in typhoon-prone zones should receive genset/battery pre-activation in August (confirmed driver is Power Failure, not FOC — see REC-P7-04).
  P7 → P8 — Breach Pred.           Temporal features (Is_HolyWeek, Is_TyphoonSeason, hour-of-day) feed the P8 ML feature matrix.


## 6. Next Steps

| Priority | Action | Timeline |
|----------|--------|----------|
| 🔴 High | Introduce Saturday 20:00 on-call rotation for top-3 breach zones | Month 1 |
| 🔴 High | Update NOC weekly report template with time-of-day and seasonal SLA splits | Month 1 |
| 🟡 Medium | Add `Is_HolyWeek` + `Is_TyphoonSeason` flags to P7 next retraining cycle | Month 2–3 |
| 🟡 Medium | Track April ticket volume for capacity planning only — no PM/staffing shift (breach rate is already below average, see REC-P7-03) | Month 3 |
| 🟡 Medium | Pre-position genset/battery support before August — confirmed driver is Power Failure, not FOC Cut (see REC-P7-04) | Month 5 |
| 🟢 Standard | Investigate Monday breach spike — backlog vs handover quality | Month 2–4 |
| ▶ Next | **Commence Project 8 — SLA Breach Prediction (ML classifier)** | Month 3 |

---

> **Project 7 complete.** Temporal decomposition (NB22–NB23) delivers the analysis deferred in NB03 Section 7c.  
> Temporal features (hour-of-day, day-of-week, Holy Week flag, typhoon season flag) are ready for inclusion in the Project 8 SLA breach prediction feature matrix.